# 🌊 Week 1, Day 4 — May 22, 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {{
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_plastic = pd.read_csv(DIRS['processed'] + '/plastic_filtered.csv')
df_species = pd.read_csv(DIRS['processed'] + '/species_filtered.csv')
df_climate = pd.read_csv(DIRS['processed'] + '/climate_flagged.csv')
df_temp    = pd.read_csv(DIRS['processed'] + '/global_temp_filtered.csv')
print('Datasets loaded ✅')

## Step 1: Missing Value Counts — All Datasets

In [ ]:
def missingness_report(df, label):
    """Print a full missingness summary for a dataframe."""
    total = len(df)
    nulls = df.isnull().sum()
    pct   = (nulls / total * 100).round(2)
    report = pd.DataFrame({'Null_Count': nulls, 'Null_Pct': pct})
    report = report[report['Null_Count'] > 0].sort_values('Null_Pct', ascending=False)
    
    print(f'\n{"="*55}')
    print(f'{label} — {total:,} rows × {df.shape[1]} cols')
    print(f'Total null cells: {nulls.sum():,} / {total * df.shape[1]:,}  ({nulls.sum()/(total*df.shape[1])*100:.1f}% sparse)')
    print(f'{"="*55}')
    if len(report) == 0:
        print('  ✅ No missing values!')
    else:
        print(f'  {"Column":<30} {"Nulls":>8} {"Pct":>8}')
        print(f'  {"-"*50}')
        for col, row in report.iterrows():
            flag = '🔴' if row['Null_Pct'] > 50 else '🟡' if row['Null_Pct'] > 10 else '🟢'
            print(f'  {flag} {col:<28} {int(row["Null_Count"]):>8,} {row["Null_Pct"]:>7.1f}%')
    return report

report_plastic = missingness_report(df_plastic, 'ocean_plastic_pollution_data')
report_species = missingness_report(df_species, 'india_cms_final_master_enriched')
report_climate = missingness_report(df_climate, 'realistic_ocean_climate_dataset')
report_temp    = missingness_report(df_temp,    'GlobalTemperatures')

## Step 2: Visual Missingness Bar Chart

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
datasets = [
    (df_species, 'india_cms (Species)', axes[0][0]),
    (df_temp,    'GlobalTemperatures',  axes[0][1]),
    (df_climate, 'Ocean Climate',       axes[1][0]),
    (df_plastic, 'Ocean Plastic',       axes[1][1]),
]

for df, title, ax in datasets:
    nulls = df.isnull().mean() * 100
    nulls = nulls[nulls > 0].sort_values(ascending=True)
    if len(nulls) == 0:
        ax.text(0.5, 0.5, '✅ No missing values', ha='center', va='center',
                transform=ax.transAxes, fontsize=13)
    else:
        colors = ['#d73027' if v > 50 else '#fc8d59' if v > 10 else '#fee090' for v in nulls]
        ax.barh(nulls.index, nulls.values, color=colors)
        ax.axvline(x=10, color='orange', linestyle='--', alpha=0.5, label='10% threshold')
        ax.axvline(x=50, color='red',    linestyle='--', alpha=0.5, label='50% threshold')
        ax.set_xlabel('Missing %')
        ax.legend(fontsize=7)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('May 22, 2026 — Missingness Matrix Across Datasets', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] + '/Day4_missingness_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Missingness chart saved ✅')

## Step 3: Classify Each Null — Handling Strategy

In [ ]:
print('MISSINGNESS HANDLING STRATEGY')
print('='*60)

strategies = [
    # (dataset, column, null_pct, strategy, reason)
    ('species', 'priority_group',    87.0, 'DROP COLUMN',        '87% null — not recoverable'),
    ('species', 'india_role',        17.4, 'FILL → "Unknown"',   'Categorical — safe to impute'),
    ('species', 'migration_season',  17.4, 'FILL → "Unknown"',   'Categorical — safe to impute'),
    ('species', 'location_type',     17.4, 'FILL → "Unknown"',   'Categorical — safe to impute'),
    ('species', 'seasonal_density',  17.4, 'FILL → "Unknown"',   'Categorical — safe to impute'),
    ('species', 'year',               0.9, 'FILL → median(year)', 'Numeric, low null rate'),
    ('species', 'month',              0.9, 'FILL → median(month)','Numeric, low null rate'),
    ('species', 'event_date',         2.6, 'FILL → NaT',          'Derive from year/month instead'),
    ('climate', 'Bleaching Severity',30.0, 'FILL → "Unknown"',   'Categorical — safe to impute'),
    ('temp',    'LandMaxTemperature',37.6, 'DROP COLUMN',         'Use LandAndOceanAvg instead'),
    ('temp',    'LandMinTemperature',37.6, 'DROP COLUMN',         'Use LandAndOceanAvg instead'),
    ('temp',    'LandAverageTemp',    0.4, 'FILL → forward-fill', 'Time series, small gap'),
]

print(f'  {"Dataset":<10} {"Column":<28} {"Null%":>6}  Strategy')
print(f'  {"-"*70}')
for ds, col, pct, strategy, reason in strategies:
    print(f'  {ds:<10} {col:<28} {pct:>5.1f}%  {strategy}')
    print(f'  {"":<10} {"":<28} {"":>6}  → {reason}')
    print()

## Step 4: Log Sparsity Statistics to Metadata

In [ ]:
log_path = DIRS['metadata'] + '/missingness_log_may22.txt'
import datetime

with open(log_path, 'w') as f:
    f.write('HCLTech Internship — Missingness Profiling Log\n')
    f.write(f'Date: May 22, 2026 | Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('='*60 + '\n\n')
    for df, label in [(df_plastic,'ocean_plastic'),(df_species,'india_species'),
                      (df_climate,'ocean_climate'),(df_temp,'global_temp')]:
        total_cells = df.shape[0] * df.shape[1]
        null_cells  = df.isnull().sum().sum()
        sparsity    = null_cells / total_cells * 100
        f.write(f'{label}: {null_cells:,} null cells / {total_cells:,} total = {sparsity:.1f}% sparse\n')
        nulls = df.isnull().sum()
        for col in nulls[nulls>0].index:
            pct = nulls[col]/len(df)*100
            f.write(f'  {col}: {nulls[col]} ({pct:.1f}%)\n')
        f.write('\n')

print(f'Missingness log saved: {log_path} ✅')